# LeetCode 2765 — Longest Alternating Subarray bằng Python

Link đề: https://leetcode.com/problems/longest-alternating-subarray/ · Độ khó: **Easy** · Chủ đề: Array, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def alternatingSubarray(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu định nghĩa **mảng con xen kẽ (alternating subarray)**: dài hơn 1, hai phần tử đầu hơn kém `+1`, rồi các hiệu số lần lượt là `+1, −1, +1, −1, ...`; tìm độ dài lớn nhất, hoặc `-1` nếu không có.
2. Dùng **hiệu hai phần tử kề nhau** `nums[i] − nums[i − 1]` để kiểm tra mẫu, và hiểu cách mẫu "dự kiến" thay đổi theo **vị trí trong đoạn** (lẻ → `+1`, chẵn → `−1`).
3. Đi từ **Brute Force O(n³)** → **mở rộng từ mỗi điểm đầu O(n²)** → **một lượt duyệt với biến `cur` (O(n))** → **hai con trỏ nhảy cóc** → **máy trạng thái hai trạng thái "đỉnh / đáy"**.
4. Biết nhánh **"khởi động lại bằng cặp mới"** khi mẫu gãy: phần tử hiện tại có thể là phần tử thứ hai của một đoạn mới.
5. Tránh các lỗi: quên trả `-1`, chấp nhận đoạn bắt đầu bằng hiệu `−1`, không khởi động lại sau khi gãy, và đòi mọi hiệu bằng nhau.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` (chỉ số từ `0`). Một mảng con `s` độ dài `m` được gọi là **xen kẽ (alternating)** nếu:

- `m` **lớn hơn 1**, và
- `s[1] = s[0] + 1`, và
- `s` có dạng `[s[0], s[1], s[0], s[1], ..., s[(m − 1) % 2]]`. Nói cách khác: `s[1] − s[0] = 1`, `s[2] − s[1] = −1`, `s[3] − s[2] = 1`, `s[4] − s[3] = −1`, ... tức là `s[i] − s[i − 1] = (−1)^(i + 1)` với `i` từ `1` tới `m − 1`.

Trả về **độ dài lớn nhất** của mọi mảng con xen kẽ trong `nums`, hoặc `-1` nếu **không có** mảng con xen kẽ nào.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[2,3,4,3,4]` | `4` | Các mảng con xen kẽ gồm `[3,4]`, `[3,4,3]`, `[3,4,3,4]`; dài nhất là `[3,4,3,4]` (độ dài 4). |
| 2 | `[4,5,6]` | `2` | `[4,5]` và `[5,6]` là các mảng con xen kẽ; cả hai dài 2. `[4,5,6]` không xen kẽ vì sau `+1` phải là `−1`. |

Constraints:

- `2 <= nums.length <= 100`
- `1 <= nums[i] <= 10^4`

In [1]:
nums = [2, 3, 4, 3, 4]
# in các hiệu giữa hai phần tử kề nhau
diffs = [nums[i] - nums[i - 1] for i in range(1, len(nums))]
print(diffs)       # [1, 1, -1, 1]
# đoạn [3,4,3,4] ứng với hiệu [1, -1, 1]
print(nums[1:5], [nums[i] - nums[i - 1] for i in range(2, 5)])

[1, 1, -1, 1]
[3, 4, 3, 4] [1, -1, 1]


## 3. Kiến thức nền

### 3.1. Hiệu hai phần tử kề nhau

Với mảng `nums`, hiệu `nums[i] − nums[i − 1]` (với `i >= 1`) cho biết số đi **lên** (`+`) hay **xuống** (`−`) bao nhiêu. Mảng xen kẽ của đề có hiệu là dãy `+1, −1, +1, −1, ...` **bắt đầu bằng `+1`**.

In [2]:
nums = [3, 4, 3, 4]
print([nums[i] - nums[i - 1] for i in range(1, len(nums))])   # [1, -1, 1]
nums = [4, 5, 6]
print([nums[i] - nums[i - 1] for i in range(1, len(nums))])   # [1, 1]  không xen kẽ

[1, -1, 1]
[1, 1]


### 3.2. Vị trí trong đoạn và dấu của hiệu

Trong một đoạn bắt đầu ở `l`, phần tử thứ `k` (đánh số từ `0` bên trong đoạn, tức vị trí `l + k`) có hiệu so với phần tử trước là:

- `+1` nếu `k` **lẻ**,
- `−1` nếu `k` **chẵn** (và `k >= 2`).

Nếu đoạn hiện đã có `cur` phần tử, thì phần tử **kế tiếp** sẽ có `k = cur`, nên hiệu dự kiến là `+1` khi `cur` lẻ và `−1` khi `cur` chẵn.

In [3]:
for cur in range(1, 7):
    expected = 1 if cur % 2 == 1 else -1
    print(f"đoạn đang có {cur} phần tử -> hiệu dự kiến của phần tử kế tiếp = {expected:+d}")

đoạn đang có 1 phần tử -> hiệu dự kiến của phần tử kế tiếp = +1
đoạn đang có 2 phần tử -> hiệu dự kiến của phần tử kế tiếp = -1
đoạn đang có 3 phần tử -> hiệu dự kiến của phần tử kế tiếp = +1
đoạn đang có 4 phần tử -> hiệu dự kiến của phần tử kế tiếp = -1
đoạn đang có 5 phần tử -> hiệu dự kiến của phần tử kế tiếp = +1
đoạn đang có 6 phần tử -> hiệu dự kiến của phần tử kế tiếp = -1


### 3.3. Công thức `(-1) ** k` (tuỳ chọn)

Trong Toán, `(-1) ** k` bằng `1` nếu `k` chẵn và `-1` nếu `k` lẻ. Hiệu dự kiến tại phần tử thứ `k` là `(-1) ** (k + 1)`. Trong code ta dùng `1 if k % 2 == 1 else -1` cho dễ đọc.

In [4]:
for k in range(1, 6):
    print(k, (-1) ** (k + 1), 1 if k % 2 == 1 else -1)

1 1 1
2 -1 -1
3 1 1
4 -1 -1
5 1 1


### 3.4. Duyệt có `enumerate` và bắt đầu từ chỉ số 1

`for i in range(1, n)` duyệt từ phần tử thứ hai để luôn có phần tử trước `nums[i − 1]`. Lưu ý mảng có tối thiểu 2 phần tử theo ràng buộc, nên vòng lặp luôn chạy ít nhất một lần.

In [5]:
nums = [2, 3, 4]
for i in range(1, len(nums)):
    print(f"i={i}: nums[{i-1}]={nums[i-1]}, nums[{i}]={nums[i]}, hiệu={nums[i] - nums[i-1]}")

i=1: nums[0]=2, nums[1]=3, hiệu=1
i=2: nums[1]=3, nums[2]=4, hiệu=1


### 3.5. Đáp án `-1` khi không tồn tại

Ta duy trì `best = 0` (hoặc `1`) và cuối cùng trả `best if best >= 2 else -1`. Đừng trả `0` hay `1`: đề quy định `-1` khi **không có** mảng con xen kẽ (một phần tử đơn lẻ **không** được tính, vì `m > 1`).

In [6]:
best = 0
print(best if best >= 2 else -1)      # -1
best = 2
print(best if best >= 2 else -1)      # 2

-1
2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Duyệt ví dụ 1: `nums = [2, 3, 4, 3, 4]`. Viết dãy hiệu `[1, 1, -1, 1]` (giữa các cặp kề nhau).

- Hiệu đầu `+1` ⇒ `[2, 3]` là mảng xen kẽ độ dài 2.
- Hiệu tiếp `+1` nhưng mẫu cần `−1` ⇒ `[2, 3, 4]` **không** xen kẽ. Tuy nhiên cặp `(3, 4)` có hiệu `+1`, nên một đoạn **mới** bắt đầu tại phần tử `3` (vị trí 1): `[3, 4]`.
- Hiệu tiếp `−1` đúng mẫu ⇒ `[3, 4, 3]` (dài 3).
- Hiệu tiếp `+1` đúng mẫu ⇒ `[3, 4, 3, 4]` (dài 4). Hết mảng.

**Quan sát then chốt:** khi mẫu gãy tại `i`, đoạn cũ kết thúc, nhưng nếu `nums[i] − nums[i − 1] = +1` thì **cặp `(nums[i − 1], nums[i])` mở một đoạn mới** (độ dài 2). Đừng bỏ sót.

In [7]:
nums = [2, 3, 4, 3, 4]
cur = 1
for i in range(1, len(nums)):
    d = nums[i] - nums[i - 1]
    expected = 1 if cur % 2 == 1 else -1
    print(f"i={i}: hiệu={d:+d}, dự kiến={expected:+d}, cur={cur}")
    if d == expected:
        cur += 1
    elif d == 1:
        cur = 2
    else:
        cur = 1
print("cur cuối:", cur)

i=1: hiệu=+1, dự kiến=+1, cur=1
i=2: hiệu=+1, dự kiến=-1, cur=2
i=3: hiệu=-1, dự kiến=-1, cur=2
i=4: hiệu=+1, dự kiến=+1, cur=3
cur cuối: 4


## 5. Cách 1 — Brute Force / baseline: kiểm tra mọi đoạn `[l, r]`

**Ý tưởng:** thử mọi cặp `(l, r)` với `l < r` (độ dài `>= 2`). Với mỗi đoạn, kiểm tra từng vị trí `k` từ `1` tới `r − l`: `nums[l + k] − nums[l + k − 1]` phải bằng `+1` nếu `k` lẻ, `−1` nếu `k` chẵn. Nếu đúng hết thì đoạn xen kẽ, cập nhật độ dài lớn nhất. Không có thì trả `-1`.

Bản này dịch thẳng đề bài nên rất đáng tin để làm **oracle**.

In [8]:
from typing import List

class SolutionBrute:
    def alternatingSubarray(self, nums: List[int]) -> int:
        n = len(nums)
        best = -1
        for l in range(n):
            for r in range(l + 1, n):
                ok = True
                for k in range(1, r - l + 1):
                    expected = 1 if k % 2 == 1 else -1
                    if nums[l + k] - nums[l + k - 1] != expected:
                        ok = False
                        break
                if ok:
                    best = max(best, r - l + 1)
        return best

In [9]:
sol = SolutionBrute()
print(sol.alternatingSubarray([2, 3, 4, 3, 4]))      # 4
print(sol.alternatingSubarray([4, 5, 6]))            # 2
print(sol.alternatingSubarray([5, 5, 5]))            # -1

4
2
-1


**Độ phức tạp:** `O(n²)` đoạn × kiểm `O(n)` ⇒ Time `O(n³)`, Space `O(1)`.

**Nút thắt:** với mỗi đoạn ta kiểm tra lại từ đầu dù đoạn `[l, r]` chỉ hơn `[l, r − 1]` đúng một phần tử. Nếu `[l, r − 1]` đã xen kẽ thì chỉ cần kiểm tra hiệu cuối.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — mở rộng dần từ mỗi điểm đầu.** Điểm đầu `l` hợp lệ khi `nums[l + 1] − nums[l] = 1`. Từ đó tăng `r` chừng nào hiệu mới khớp dự kiến; gặp lỗi thì dừng (đoạn dài hơn chứa chỗ lỗi). Cho ra `O(n²)`.

**Đột phá 2 — chỉ nhớ "độ dài hiện tại".** Gọi `cur` là độ dài của mảng xen kẽ **dài nhất kết thúc tại vị trí đang xét** (với quy ước `cur = 1` nghĩa là chưa có cặp hợp lệ, chỉ có phần tử đó). Khi tới `i`, tính `d = nums[i] − nums[i − 1]` và hiệu dự kiến `e = +1 nếu cur lẻ, −1 nếu cur chẵn`:

1. Nếu `d == e` ⇒ `cur += 1` (kéo dài đoạn hiện tại).
2. Nếu không, nhưng `d == +1` ⇒ `cur = 2` (cặp mới `(nums[i − 1], nums[i])` làm đoạn mới).
3. Ngược lại ⇒ `cur = 1` (không có đoạn nào kết thúc ở `i`).

Cập nhật `best = max(best, cur)`; cuối cùng trả `best` nếu `best >= 2`, ngược lại `-1`.

**Đột phá 3 — nhảy cóc.** Khi đoạn bắt đầu ở `l` kết thúc ở `r` (vì hiệu tiếp theo sai), các điểm đầu nằm trong `(l, r)` chỉ cho đoạn **ngắn hơn** cùng dừng tại `r`. Điểm đầu có ý nghĩa tiếp theo là `r` (cặp `(nums[r], nums[r + 1])` có thể là `+1`). Vậy ta nhảy `l = r`.

**Đột phá 4 — hai trạng thái.** Thay vì đếm `cur` và suy ra dự kiến từ tính chẵn lẻ, giữ hai biến: `up` = độ dài đoạn kết thúc bằng một bước `+1` ("đỉnh"), `down` = độ dài đoạn kết thúc bằng bước `−1` ("đáy"). Bước `+1` kéo từ `down` (hoặc mở cặp mới nếu `down = 0`); bước `−1` chỉ kéo từ `up`.

In [10]:
# Kiểm tra nhanh quy luật khởi động lại với [1, 2, 3, 2]
nums = [1, 2, 3, 2]
cur, best = 1, 0
for i in range(1, len(nums)):
    d = nums[i] - nums[i - 1]
    e = 1 if cur % 2 == 1 else -1
    if d == e:
        cur += 1; why = "khớp mẫu"
    elif d == 1:
        cur = 2; why = "gãy nhưng hiệu +1 -> mở đoạn mới"
    else:
        cur = 1; why = "gãy"
    best = max(best, cur)
    print(f"i={i}: d={d:+d}, e={e:+d}, cur={cur}, best={best}  ({why})")

i=1: d=+1, e=+1, cur=2, best=2  (khớp mẫu)
i=2: d=+1, e=-1, cur=2, best=2  (gãy nhưng hiệu +1 -> mở đoạn mới)
i=3: d=-1, e=-1, cur=3, best=3  (khớp mẫu)


## 7. Các cách tiếp theo

### Cách 2 — Mở rộng từ mỗi điểm đầu hợp lệ

**Từng đoạn:**

- Với mỗi `l`, nếu `l + 1 < n` và `nums[l + 1] − nums[l] == 1` thì đây là điểm đầu hợp lệ.
- Đặt `r = l + 1`; vòng `while r + 1 < n` kiểm tra hiệu tiếp theo `nums[r + 1] − nums[r]` có bằng `+1` nếu `(r + 1 − l)` lẻ, `−1` nếu chẵn không; nếu có thì `r += 1`, ngược lại `break`.
- Độ dài `r − l + 1`.

In [11]:
class SolutionExpand:
    def alternatingSubarray(self, nums: List[int]) -> int:
        n = len(nums)
        best = -1
        for l in range(n - 1):
            if nums[l + 1] - nums[l] != 1:
                continue
            r = l + 1
            while r + 1 < n:
                k = r + 1 - l                           # vị trí của phần tử mới trong đoạn
                expected = 1 if k % 2 == 1 else -1
                if nums[r + 1] - nums[r] != expected:
                    break
                r += 1
            best = max(best, r - l + 1)
        return best

In [12]:
sol = SolutionExpand()
print(sol.alternatingSubarray([2, 3, 4, 3, 4]))      # 4
print(sol.alternatingSubarray([4, 5, 6]))            # 2
print(sol.alternatingSubarray([5, 5, 5]))            # -1

4
2
-1


**Độ phức tạp:** Time `O(n²)`, Space `O(1)`.

### Cách 3 — Một lượt duyệt với biến `cur` (code nộp)

**Từng đoạn:** đúng ba nhánh ở mục 6. `cur` bắt đầu bằng `1`; mỗi `i` từ `1` tới `n − 1`:

- `e = 1 if cur % 2 == 1 else -1`
- `d == e` → kéo dài; `d == 1` → `cur = 2`; còn lại `cur = 1`.

In [13]:
class SolutionCur:
    def alternatingSubarray(self, nums: List[int]) -> int:
        cur = 1
        best = 0
        for i in range(1, len(nums)):
            d = nums[i] - nums[i - 1]
            expected = 1 if cur % 2 == 1 else -1
            if d == expected:
                cur += 1
            elif d == 1:
                cur = 2
            else:
                cur = 1
            best = max(best, cur)
        return best if best >= 2 else -1

In [14]:
sol = SolutionCur()
print(sol.alternatingSubarray([2, 3, 4, 3, 4]))      # 4
print(sol.alternatingSubarray([4, 5, 6]))            # 2
print(sol.alternatingSubarray([5, 5, 5]))            # -1
print(sol.alternatingSubarray([1, 2, 3, 2]))         # 3

4
2
-1
3


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 4 — Hai con trỏ, nhảy `l = r`

**Từng đoạn:**

- `l` duyệt từ `0`. Nếu `l + 1 < n` và hiệu `nums[l + 1] − nums[l] == 1`: mở rộng `r` như Cách 2 (chỉ vòng `while`), ghi nhận `r − l + 1`, rồi **nhảy** `l = r`.
- Ngược lại `l += 1`.

Lý do nhảy được: xem mục 6, đột phá 3.

In [15]:
class SolutionJump:
    def alternatingSubarray(self, nums: List[int]) -> int:
        n = len(nums)
        best = -1
        l = 0
        while l + 1 < n:
            if nums[l + 1] - nums[l] != 1:
                l += 1
                continue
            r = l + 1
            while r + 1 < n:
                k = r + 1 - l
                expected = 1 if k % 2 == 1 else -1
                if nums[r + 1] - nums[r] != expected:
                    break
                r += 1
            best = max(best, r - l + 1)
            l = r                       # r >= l + 1 nên chắc chắn tiến lên
        return best

In [16]:
sol = SolutionJump()
print(sol.alternatingSubarray([2, 3, 4, 3, 4]))      # 4
print(sol.alternatingSubarray([4, 5, 6]))            # 2
print(sol.alternatingSubarray([5, 5, 5]))            # -1
print(sol.alternatingSubarray([1, 2, 3, 2]))         # 3

4
2
-1
3


**Độ phức tạp:** mỗi bước của `r` tiến chỉ số, `l` nhảy tới `r`, nên mỗi chỉ số được thăm `O(1)` lần ⇒ Time `O(n)`, Space `O(1)`.

### Cách 5 — Máy trạng thái hai trạng thái `up` / `down`

**Từng đoạn:**

- `up`: độ dài lớn nhất của đoạn xen kẽ kết thúc tại `i` mà bước cuối là `+1` (phần tử "đỉnh"). `down`: tương tự với bước cuối `−1` ("đáy").
- Hiệu `d = nums[i] − nums[i − 1]`:
  - `d == 1`: `new_up = down + 1 nếu down > 0, ngược lại 2`; `new_down = 0`.
  - `d == -1`: `new_down = up + 1 nếu up > 0, ngược lại 0`; `new_up = 0`.
  - còn lại: cả hai `0`.
- `best = max(best, up, down)`; cuối cùng `best` nếu `>= 2` ngược lại `-1`.

Chú ý `down = 0` thể hiện "chưa có đoạn kết thúc ở đáy" nên bước `+1` mở cặp mới (độ dài 2).

In [17]:
class SolutionUpDown:
    def alternatingSubarray(self, nums: List[int]) -> int:
        up = down = 0
        best = 0
        for i in range(1, len(nums)):
            d = nums[i] - nums[i - 1]
            if d == 1:
                up, down = (down + 1 if down > 0 else 2), 0
            elif d == -1:
                up, down = 0, (up + 1 if up > 0 else 0)
            else:
                up = down = 0
            best = max(best, up, down)
        return best if best >= 2 else -1

In [18]:
sol = SolutionUpDown()
print(sol.alternatingSubarray([2, 3, 4, 3, 4]))      # 4
print(sol.alternatingSubarray([4, 5, 6]))            # 2
print(sol.alternatingSubarray([5, 5, 5]))            # -1
print(sol.alternatingSubarray([1, 2, 3, 2]))         # 3

4
2
-1
3


**Độ phức tạp:** Time `O(n)`, Space `O(1)`. Cách này nhìn rõ hơn "mẫu `+1, −1`" thành hai trạng thái, không cần suy dự kiến từ tính chẵn lẻ của `cur`.

## 8. TRACE từng bước

Trace Cách 3 trên `nums = [2, 3, 4, 3, 4]`:

| i | `nums[i−1]` → `nums[i]` | `d` | `cur` trước | dự kiến `e` | quyết định | `cur` sau | `best` |
|---|---|---|---|---|---|---|---|
| 1 | 2 → 3 | +1 | 1 | +1 | khớp, kéo dài | 2 | 2 |
| 2 | 3 → 4 | +1 | 2 | −1 | không khớp, nhưng `d = +1` ⇒ mở đoạn mới | 2 | 2 |
| 3 | 4 → 3 | −1 | 2 | −1 | khớp, kéo dài | 3 | 3 |
| 4 | 3 → 4 | +1 | 3 | +1 | khớp, kéo dài | 4 | 4 |

Trả `4`. Chú ý ở `i = 2`: nếu ta chỉ đặt `cur = 1` ta sẽ bỏ lỡ đoạn `[3, 4, 3, 4]` bắt đầu từ vị trí 1.

In [19]:
def trace_cur(nums):
    cur, best = 1, 0
    for i in range(1, len(nums)):
        d = nums[i] - nums[i - 1]
        e = 1 if cur % 2 == 1 else -1
        before = cur
        if d == e:
            cur += 1; why = "khớp -> kéo dài"
        elif d == 1:
            cur = 2; why = "gãy nhưng d=+1 -> đoạn mới"
        else:
            cur = 1; why = "gãy -> cur=1"
        best = max(best, cur)
        print(f"i={i}, {nums[i-1]}->{nums[i]}, d={d:+d}, e={e:+d}, cur: {before}->{cur}, best={best}   ({why})")
    ans = best if best >= 2 else -1
    print("kết quả:", ans)
    return ans

trace_cur([2, 3, 4, 3, 4])
trace_cur([4, 5, 6])
trace_cur([7, 7, 7])

i=1, 2->3, d=+1, e=+1, cur: 1->2, best=2   (khớp -> kéo dài)
i=2, 3->4, d=+1, e=-1, cur: 2->2, best=2   (gãy nhưng d=+1 -> đoạn mới)
i=3, 4->3, d=-1, e=-1, cur: 2->3, best=3   (khớp -> kéo dài)
i=4, 3->4, d=+1, e=+1, cur: 3->4, best=4   (khớp -> kéo dài)
kết quả: 4
i=1, 4->5, d=+1, e=+1, cur: 1->2, best=2   (khớp -> kéo dài)
i=2, 5->6, d=+1, e=-1, cur: 2->2, best=2   (gãy nhưng d=+1 -> đoạn mới)
kết quả: 2
i=1, 7->7, d=+0, e=+1, cur: 1->1, best=1   (gãy -> cur=1)
i=2, 7->7, d=+0, e=+1, cur: 1->1, best=1   (gãy -> cur=1)
kết quả: -1


-1

In [20]:
def trace_updown(nums):
    up = down = best = 0
    for i in range(1, len(nums)):
        d = nums[i] - nums[i - 1]
        if d == 1:
            up, down = (down + 1 if down > 0 else 2), 0
        elif d == -1:
            up, down = 0, (up + 1 if up > 0 else 0)
        else:
            up = down = 0
        best = max(best, up, down)
        print(f"i={i}, d={d:+d}, up={up}, down={down}, best={best}")
    return best if best >= 2 else -1

print(trace_updown([2, 3, 4, 3, 4]))

i=1, d=+1, up=2, down=0, best=2
i=2, d=+1, up=2, down=0, best=2
i=3, d=-1, up=0, down=3, best=3
i=4, d=+1, up=4, down=0, best=4
4


## 9. Code nộp LeetCode

Dùng Cách 3 (biến `cur`).

In [21]:
from typing import List

class Solution:
    def alternatingSubarray(self, nums: List[int]) -> int:
        cur = 1        # độ dài đoạn xen kẽ dài nhất kết thúc tại vị trí đang xét (1 = chưa có cặp)
        best = 0
        for i in range(1, len(nums)):
            d = nums[i] - nums[i - 1]
            expected = 1 if cur % 2 == 1 else -1     # hiệu cần có để kéo dài
            if d == expected:
                cur += 1                              # kéo dài đoạn
            elif d == 1:
                cur = 2                               # gãy, nhưng (nums[i-1], nums[i]) mở đoạn mới
            else:
                cur = 1                               # không có đoạn nào kết thúc ở i
            best = max(best, cur)
        return best if best >= 2 else -1

In [22]:
print(Solution().alternatingSubarray([2, 3, 4, 3, 4]))      # 4
print(Solution().alternatingSubarray([4, 5, 6]))            # 2

4
2


## 10. Vì sao thuật toán đúng

**Định nghĩa.** Sau khi xử lý vị trí `i`, `cur_i` là độ dài lớn nhất của **mảng con xen kẽ kết thúc ở `i`** nếu có (độ dài `>= 2`); nếu không có thì `cur_i = 1`.

**Nhận xét về cấu trúc.** Mảng con xen kẽ độ dài `m >= 2` kết thúc ở `i` có hiệu cuối `d = nums[i] − nums[i − 1]` bằng `+1` nếu `m` chẵn (vì phần tử cuối có `k = m − 1` lẻ) và `−1` nếu `m` lẻ. Bỏ phần tử cuối ta được mảng con xen kẽ độ dài `m − 1` kết thúc ở `i − 1` (nếu `m − 1 >= 2`).

**Bổ đề (chỉ có các độ dài cùng tính chẵn lẻ).** Nếu mảng con xen kẽ độ dài `L` kết thúc ở `i`, thì phần hậu tố bắt đầu tại offset `j` (trong đoạn) chỉ xen kẽ khi bước đầu của nó (bước ở offset `j + 1`) là `+1`, tức `j` **chẵn**. Vì vậy mọi độ dài xen kẽ kết thúc ở `i` có dạng `L, L − 2, L − 4, ...` (`>= 2`), **cùng tính chẵn lẻ** với `L` lớn nhất. Hệ quả: hiệu dự kiến để kéo dài từ một đoạn kết thúc ở `i − 1` chỉ phụ thuộc vào tính chẵn lẻ của `cur_{i−1}` (chính là `e` trong code), không phụ thuộc vào đoạn nào trong họ.

**Quy nạp.**

1. **Kéo dài (`d == e`).** Thêm `nums[i]` vào đoạn dài nhất kết thúc ở `i − 1` cho đoạn xen kẽ dài `cur_{i−1} + 1`. Không thể có đoạn dài hơn: nếu có đoạn xen kẽ độ dài `m' > cur_{i−1} + 1` kết thúc ở `i` (với `m' >= 3`), bỏ phần tử cuối ta được đoạn xen kẽ độ dài `m' − 1 > cur_{i−1}` kết thúc ở `i − 1`, mâu thuẫn với tính lớn nhất của `cur_{i−1}`. ✓
2. **Gãy nhưng `d == +1`.** Giả sử có đoạn xen kẽ độ dài `m >= 3` kết thúc ở `i`. Bỏ phần tử cuối ta được đoạn xen kẽ độ dài `m − 1 >= 2` kết thúc ở `i − 1`; theo bổ đề `m − 1` cùng tính chẵn lẻ với `cur_{i−1}`, nên hiệu dự kiến `e(m − 1) = e(cur_{i−1}) = e`, và `d` phải bằng `e`: mâu thuẫn với "gãy". Vậy không có đoạn độ dài `>= 3`; còn đoạn độ dài `2` là `(nums[i − 1], nums[i])` hợp lệ vì `d = +1`. Do đó `cur_i = 2`. ✓
3. **Ngược lại (`d != e` và `d != +1`).** Đoạn độ dài `2` cần `d = +1` nên không có; đoạn độ dài `>= 3` bị loại như trên. Vậy không có đoạn nào kết thúc ở `i`, `cur_i = 1`. ✓

Đáp án là `max_i cur_i` nếu `>= 2`, vì mọi đoạn xen kẽ kết thúc ở một `i` nào đó. Không có đoạn nào thì `best` luôn `1`, ta trả `-1`.

Cách 1 là bản dịch thẳng của đề; test ở mục 12 chứng minh bằng thực nghiệm các cách còn lại khớp với nó.

## 11. Phân tích Time / Space Complexity

`n = len(nums)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n³)` | `O(1)` | kiểm từng đoạn từ đầu |
| 2. Mở rộng từ mỗi điểm đầu | `O(n²)` | `O(1)` | dừng ngay khi gãy |
| 3. Biến `cur` | `O(n)` | `O(1)` | **code nộp** |
| 4. Hai con trỏ nhảy cóc | `O(n)` | `O(1)` | mỗi chỉ số thăm `O(1)` lần |
| 5. `up` / `down` | `O(n)` | `O(1)` | hai trạng thái rõ ràng |

Với `n <= 100` mọi cách đều rất nhanh, nhưng phân tích giúp hiểu **tại sao** có thể giảm từ `n³` xuống `n`: mỗi hiệu `nums[i] − nums[i−1]` chỉ cần xét **một lần**.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: hai phần tử (đúng `+1` / sai); toàn bằng nhau (`-1`); giảm dần (`-1`); xen kẽ hoàn hảo dài; xen kẽ rồi gãy rồi lại xen kẽ; cần khởi động lại sau khi gãy (`[1,2,3,2]`); bắt đầu bằng `−1` (không hợp lệ làm đầu).

In [23]:
sols = [SolutionBrute(), SolutionExpand(), SolutionCur(), SolutionJump(), SolutionUpDown(), Solution()]
cases = [
    ("ví dụ 1", [2, 3, 4, 3, 4], 4),
    ("ví dụ 2", [4, 5, 6], 2),
    ("hai phần tử +1", [1, 2], 2),
    ("hai phần tử -1", [2, 1], -1),
    ("hai phần tử bằng nhau", [3, 3], -1),
    ("toàn bằng nhau", [5, 5, 5, 5], -1),
    ("giảm dần", [5, 4, 3, 2, 1], -1),
    ("xen kẽ hoàn hảo", [1, 2, 1, 2, 1, 2, 1], 7),
    ("xen kẽ hoàn hảo, bắt đầu giảm", [2, 1, 2, 1, 2], 4),
    ("cần khởi động lại", [1, 2, 3, 2], 3),
    ("gãy rồi lại xen kẽ", [1, 2, 1, 5, 6, 5, 6, 5], 5),
    ("hiệu lớn hơn 1", [1, 3, 1, 3], -1),
    ("tăng dần liên tục", [1, 2, 3, 4, 5], 2),
    ("bắt đầu -1 rồi +1", [3, 2, 3], 2),
    ("số lớn", [10000, 9999, 10000, 9999, 10000], 4),
]
for name, nums, exp in cases:
    for s in sols:
        got = s.alternatingSubarray(nums[:])
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 4
PASS ví dụ 2 -> 2
PASS hai phần tử +1 -> 2
PASS hai phần tử -1 -> -1
PASS hai phần tử bằng nhau -> -1
PASS toàn bằng nhau -> -1
PASS giảm dần -> -1
PASS xen kẽ hoàn hảo -> 7
PASS xen kẽ hoàn hảo, bắt đầu giảm -> 4
PASS cần khởi động lại -> 3
PASS gãy rồi lại xen kẽ -> 5
PASS hiệu lớn hơn 1 -> -1
PASS tăng dần liên tục -> 2
PASS bắt đầu -1 rồi +1 -> 2
PASS số lớn -> 4


Giải thích vài ca:

- `[2, 1, 2, 1, 2]`: không thể bắt đầu ở `2` đầu tiên (hiệu `−1`); bắt đầu ở vị trí 1: `[1, 2, 1, 2]` dài 4.
- `[1, 2, 1, 5, 6, 5, 6, 5]`: đoạn `[1, 2, 1]` (3), sau đó `1 → 5` gãy; `[5, 6, 5, 6, 5]` dài 5.
- `[3, 2, 3]`: bắt đầu ở `3` (hiệu `−1`) không hợp lệ; `[2, 3]` dài 2. Một lời giải sai nhận `3`.

### 12.2. Đối chiếu ngẫu nhiên (hợp lệ theo đề)

Sinh `n` trong `2..100`, giá trị trong `1..10^4`. Để có nhiều đoạn xen kẽ, ta dùng ba kiểu sinh: giá trị nhỏ (`1..4`), "bước đi ngẫu nhiên" ±1 quanh một giá trị, và các đoạn xen kẽ hoàn hảo ghép lại với nhiễu.

In [24]:
import random
random.seed(2765)

def gen_small():
    n = random.randint(2, 100)
    hi = random.choice([2, 3, 4, 5])
    return [random.randint(1, hi) for _ in range(n)]

def gen_walk():
    n = random.randint(2, 100)
    x = random.randint(2, 9999)
    nums = [x]
    for _ in range(n - 1):
        x = min(10000, max(1, x + random.choice([-1, 1, 1, -1, 0, 2])))
        nums.append(x)
    return nums

def gen_pieces():
    nums = []
    while len(nums) < 100 and random.random() < 0.97:
        a = random.randint(1, 9999)
        L = random.randint(1, 12)
        piece = [a + (i % 2) for i in range(L)]                 # a, a+1, a, a+1, ...
        if random.random() < 0.3:
            piece[-1] += random.choice([-2, 2, 5])              # phá hỏng cuối đoạn
        nums.extend(max(1, min(10000, v)) for v in piece)
    nums = nums[:100]
    return nums if len(nums) >= 2 else [1, 2]

count = 0
for _ in range(400):
    for gen in (gen_small, gen_walk, gen_pieces):
        nums = gen()
        assert 2 <= len(nums) <= 100 and all(1 <= v <= 10000 for v in nums)
        exp = SolutionBrute().alternatingSubarray(nums[:])
        for s in sols:
            got = s.alternatingSubarray(nums[:])
            assert got == exp, (type(s).__name__, nums, got, exp)
        snap = nums[:]
        Solution().alternatingSubarray(nums)
        assert nums == snap
        count += 1
print(count, "ca ngẫu nhiên: 6 cách khớp nhau, giữ nguyên dữ liệu vào")

1200 ca ngẫu nhiên: 6 cách khớp nhau, giữ nguyên dữ liệu vào


### 12.3. Một phản ví dụ chống lỗi "đòi mọi hiệu bằng nhau"

Với `[4, 5, 6]`, hai hiệu đều `+1` nhưng đáp án là `2` chứ không phải `3`. Test này nằm trong bộ test chính; mục 13 giải thích chi tiết.

In [25]:
assert Solution().alternatingSubarray([4, 5, 6]) == 2
assert Solution().alternatingSubarray([1, 2, 3, 4, 5, 6]) == 2
print("đúng: tăng dần liên tục chỉ cho đoạn dài 2")

đúng: tăng dần liên tục chỉ cho đoạn dài 2


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Trả `0` hoặc `1` thay vì `-1` khi không có đoạn nào

In [26]:
def sai_khong_tra_tru_mot(nums: List[int]) -> int:
    cur, best = 1, 0
    for i in range(1, len(nums)):
        d = nums[i] - nums[i - 1]
        e = 1 if cur % 2 == 1 else -1
        if d == e:
            cur += 1
        elif d == 1:
            cur = 2
        else:
            cur = 1
        best = max(best, cur)
    return best                           # sai: thiếu "if best >= 2 else -1"

print("sai :", sai_khong_tra_tru_mot([5, 5, 5]))                # 1 (best bị kẹt ở 1)
print("đúng:", Solution().alternatingSubarray([5, 5, 5]))         # -1

sai : 1
đúng: -1


**Vì sao sai:** đề yêu cầu `-1` khi **không có** mảng con xen kẽ. Một phần tử đơn lẻ không tính (cần `m > 1`).

### Lỗi 2: Không khởi động lại sau khi gãy

In [27]:
def sai_khong_khoi_dong_lai(nums: List[int]) -> int:
    cur, best = 1, 0
    for i in range(1, len(nums)):
        d = nums[i] - nums[i - 1]
        e = 1 if cur % 2 == 1 else -1
        if d == e:
            cur += 1
        else:
            cur = 1                       # sai: bỏ qua trường hợp d == +1 mở đoạn mới
        best = max(best, cur)
    return best if best >= 2 else -1

print("sai :", sai_khong_khoi_dong_lai([1, 2, 3, 2]))             # 2
print("đúng:", Solution().alternatingSubarray([1, 2, 3, 2]))       # 3

sai : 2
đúng: 3


**Vì sao sai:** ở `i = 2` (`2 → 3`, `d = +1`) mẫu cũ gãy, nhưng `(2, 3)` có hiệu `+1` nên là một đoạn mới, và nó tiếp tục thành `[2, 3, 2]` (dài 3). Reset về `1` làm mất cơ hội này.

### Lỗi 3: Cho phép điểm đầu có hiệu `−1`

In [28]:
def sai_dau_tru_mot(nums: List[int]) -> int:
    n, best = len(nums), -1
    for l in range(n - 1):
        if abs(nums[l + 1] - nums[l]) != 1:          # sai: chấp nhận cả -1 làm hiệu đầu
            continue
        r = l + 1
        while r + 1 < n and nums[r + 1] - nums[r] == -(nums[r] - nums[r - 1]):
            r += 1
        best = max(best, r - l + 1)
    return best

print("sai :", sai_dau_tru_mot([3, 2, 3]))                      # 3
print("đúng:", Solution().alternatingSubarray([3, 2, 3]))        # 2

sai : 3
đúng: 2


**Vì sao sai:** đề quy định `s[1] = s[0] + 1`, tức hiệu đầu **phải** là `+1` (đoạn tăng trước rồi giảm). Mảng `[3, 2, 3]` bắt đầu bằng `−1` nên không hợp lệ; chỉ `[2, 3]` hợp lệ (dài 2).

### Lỗi 4: Đòi mọi hiệu bằng nhau (hoặc chỉ cần "hơn kém 1")

In [29]:
def sai_moi_hieu_bang_nhau(nums: List[int]) -> int:
    cur = best = 1
    for i in range(1, len(nums)):
        if abs(nums[i] - nums[i - 1]) == 1:          # sai: không kiểm dấu xen kẽ
            cur += 1
        else:
            cur = 1
        best = max(best, cur)
    return best if best >= 2 else -1

print("sai :", sai_moi_hieu_bang_nhau([4, 5, 6]))              # 3
print("đúng:", Solution().alternatingSubarray([4, 5, 6]))        # 2

sai : 3
đúng: 2


**Vì sao sai:** sau bước `+1` phải là bước `−1`. `[4, 5, 6]` có hai bước `+1` liền nhau nên không xen kẽ.

### Lỗi 5: Quên rằng đoạn phải liên tục (mảng con, không phải dãy con)

Mảng con là đoạn **liên tiếp**. Từ `[1, 2, 9, 1, 2]` ta **không** được bỏ `9` để ghép `[1, 2, 1, 2]`.

In [30]:
print(Solution().alternatingSubarray([1, 2, 9, 1, 2]))     # 2, không phải 4

2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Điều kiện kiểu "các hiệu liên tiếp theo mẫu cố định" → nghĩ tới **mảng hiệu** `d[i] = nums[i] − nums[i − 1]` và bài toán "đoạn liên tiếp dài nhất của `d` theo mẫu".
- Khung `cur`: `if khớp_mẫu: cur += 1; elif có_thể_mở_đoạn_mới: cur = độ_dài_tối_thiểu; else: cur = 1`; `best = max(best, cur)`. Giống bài **978 Longest Turbulent Subarray**, **2760**, **485 Max Consecutive Ones**.
- Nhớ nhánh **khởi động lại**: phần tử gây gãy mẫu thường vẫn là phần tử của một đoạn mới.
- Trả `-1` khi không tồn tại: đặt `best` ban đầu là `0` hay `1`, cuối cùng `best if best >= 2 else -1`; hoặc khởi `best = -1` và chỉ cập nhật khi `cur >= 2`.
- Test luôn gồm: hai phần tử, tất cả bằng nhau, tăng dần liên tục, xen kẽ hoàn hảo, bắt đầu ngay bằng `−1`.

## 15. Biến thể của bài

1. **978 Longest Turbulent Subarray:** dấu `<, >, <, >, ...` hoặc `>, <, >, <, ...` (bất kỳ chênh lệch nào, hai hướng bắt đầu đều được).
2. **Cho phép bắt đầu bằng `−1`** (xen kẽ ±1 hai hướng): chạy thuật toán hai lần (với `nums` và với `-nums`) hoặc sửa nhánh khởi động lại để nhận cả `d == −1`.
3. **Chênh lệch `k` thay vì `1`:** thay `1` bằng `k`.
4. **Đếm số mảng con xen kẽ:** tại mỗi `i`, số đoạn xen kẽ kết thúc ở `i` là `cur // 2` nếu `d = +1`, `(cur − 1) // 2` nếu `d = −1` (xem bài tập 5b), dựa trên bổ đề: các độ dài hợp lệ cùng tính chẵn lẻ.
5. **Trả về chính mảng con** hoặc chỉ số đầu/cuối.
6. **Xen kẽ chẵn–lẻ** (bài 2760): điều kiện trên tính chẵn lẻ chứ không phải hiệu cố định.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | kiểm mọi đoạn | `O(n³)` | `O(1)` | oracle |
| 2. Mở rộng từ điểm đầu | mỗi `l` hợp lệ, tăng `r` | `O(n²)` | `O(1)` | dễ nghĩ |
| 3. Biến `cur` | độ dài đoạn kết thúc tại `i` | `O(n)` | `O(1)` | **code nộp** |
| 4. Nhảy cóc `l = r` | tránh điểm đầu thừa | `O(n)` | `O(1)` | cần chỉ số đầu/cuối |
| 5. `up` / `down` | hai trạng thái pha | `O(n)` | `O(1)` | khi mẫu có nhiều pha |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: các hiệu của `[1, 2, 1, 2]`, hiệu dự kiến khi `cur = 2`, kết quả `best if best >= 2 else -1` với `best = 1`.

**Bài 2 (tính tay).** Với `nums = [5, 6, 7, 6, 7, 8]`: viết `d`, `e`, `cur`, `best` từng bước theo Cách 3.

**Bài 3 (điền TODO).** Hoàn thành `alternating_todo(nums)`.

In [31]:
def alternating_todo(nums):
    cur, best = 1, 0
    for i in range(1, len(nums)):
        d = nums[i] - nums[i - 1]
        # TODO 1: e = 1 nếu cur lẻ, -1 nếu cur chẵn
        # TODO 2: nếu d == e thì cur += 1; elif d == 1 thì cur = 2; else cur = 1
        best = max(best, cur)
    # TODO 3: trả best nếu best >= 2, ngược lại -1
    return None

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `alternating_updown(nums)` theo Cách 5 (hai trạng thái) mà không nhìn code mẫu. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `alt_both_dirs(nums)`: cho phép đoạn bắt đầu bằng `−1` **hoặc** `+1` (xen kẽ ±1 hai hướng). (b) Viết `count_alternating(nums)`: đếm **số** mảng con xen kẽ (độ dài `>= 2`) theo định nghĩa gốc (bắt đầu bằng `+1`); kiểm tra bằng brute force.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1, -1, 1]`; khi `cur = 2` dự kiến `−1`; `best = 1` ⇒ `-1`. Bài 2: hiệu `[1, 1, -1, 1, 1]`; đoạn `[6, 7, 6, 7]` dài 4.
- **Mức 2 (vừa):** Bài 3: `e = 1 if cur % 2 == 1 else -1`; ba nhánh như code nộp; cuối `return best if best >= 2 else -1`. Bài 4: `d == 1` ⇒ `up, down = (down + 1 if down else 2), 0`; `d == -1` ⇒ `up, down = 0, (up + 1 if up else 0)`.
- **Mức 3 (sâu):** Bài 5(a): chạy Cách 3 trên `nums` rồi trên mảng đối `[-x for x in nums]` (hiệu đổi dấu) và lấy max (cẩn thận ràng buộc `>= 1` không còn nhưng thuật toán không cần). Bài 5(b): mỗi đoạn xen kẽ kết thúc ở `i` có độ dài `L` đóng góp các hậu tố có phần tử đầu ở pha `+1`; đếm bằng cách tại mỗi `i` cộng số độ dài `m` trong `[2, cur]` có `m` cùng tính chẵn lẻ phù hợp với `d`: nếu `d = +1` thì `m` chẵn, nếu `d = −1` thì `m` lẻ `>= 3`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert [[1, 2, 1, 2][i] - [1, 2, 1, 2][i - 1] for i in range(1, 4)] == [1, -1, 1]
assert (1 if 2 % 2 == 1 else -1) == -1
best = 1
assert (best if best >= 2 else -1) == -1

# Bài 2
nums2 = [5, 6, 7, 6, 7, 8]
assert Solution().alternatingSubarray(nums2) == 4         # [6, 7, 6, 7]

# Bài 3
def alternating_todo_ans(nums):
    cur, best = 1, 0
    for i in range(1, len(nums)):
        d = nums[i] - nums[i - 1]
        e = 1 if cur % 2 == 1 else -1
        if d == e:
            cur += 1
        elif d == 1:
            cur = 2
        else:
            cur = 1
        best = max(best, cur)
    return best if best >= 2 else -1

assert alternating_todo_ans([2, 3, 4, 3, 4]) == 4
assert alternating_todo_ans([4, 5, 6]) == 2

# Bài 4
def alternating_updown(nums):
    up = down = best = 0
    for i in range(1, len(nums)):
        d = nums[i] - nums[i - 1]
        if d == 1:
            up, down = (down + 1 if down else 2), 0
        elif d == -1:
            up, down = 0, (up + 1 if up else 0)
        else:
            up = down = 0
        best = max(best, up, down)
    return best if best >= 2 else -1

assert alternating_updown([2, 3, 4, 3, 4]) == 4
assert alternating_updown([4, 5, 6]) == 2

# Bài 5(a): cho phép bắt đầu bằng -1 hoặc +1 (xen kẽ ±1 hai hướng)
def alt_both_dirs(nums):
    a = Solution().alternatingSubarray(nums)
    b = Solution().alternatingSubarray([-x for x in nums])
    return max(a, b)

assert alt_both_dirs([3, 2, 3]) == 3
assert alt_both_dirs([2, 3, 4, 3, 4]) == 4
assert alt_both_dirs([5, 5]) == -1

# Bài 5(b): đếm số mảng con xen kẽ theo định nghĩa gốc
def count_alternating(nums):
    total = 0
    cur = 1
    for i in range(1, len(nums)):
        d = nums[i] - nums[i - 1]
        e = 1 if cur % 2 == 1 else -1
        if d == e:
            cur += 1
        elif d == 1:
            cur = 2
        else:
            cur = 1
        if cur >= 2:
            # các đoạn kết thúc ở i: độ dài m trong [2, cur] với m chẵn nếu d = +1, m lẻ (>=3) nếu d = -1
            if d == 1:
                total += cur // 2
            else:
                total += (cur - 1) // 2
    return total

def brute_count(nums):
    n, c = len(nums), 0
    for l in range(n):
        for r in range(l + 1, n):
            if all(nums[l + k] - nums[l + k - 1] == (1 if k % 2 == 1 else -1) for k in range(1, r - l + 1)):
                c += 1
    return c

assert count_alternating([2, 3, 4, 3, 4]) == brute_count([2, 3, 4, 3, 4]) == 5
random.seed(11)
for _ in range(300):
    arr = random.choice([gen_small, gen_walk, gen_pieces])()
    assert count_alternating(arr) == brute_count(arr), arr
    assert alt_both_dirs(arr) >= Solution().alternatingSubarray(arr)
    assert alternating_updown(arr) == Solution().alternatingSubarray(arr)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đoạn liên tiếp theo mẫu hiệu; có nhánh khởi động lại)

```python
from typing import List

class Solution:
    def alternatingSubarray(self, nums: List[int]) -> int:
        cur, best = 1, 0
        for i in range(1, len(nums)):
            d = nums[i] - nums[i - 1]
            e = 1 if cur % 2 == 1 else -1       # hiệu cần có để kéo dài
            if d == e:
                cur += 1                        # khớp mẫu
            elif d == 1:
                cur = 2                         # gãy, nhưng cặp mới bắt đầu bằng +1
            else:
                cur = 1
            best = max(best, cur)
        return best if best >= 2 else -1
```

**Checklist**

- [ ] Hiệu đầu **phải** là `+1`, rồi `−1`, `+1`, ...?
- [ ] Hiệu dự kiến suy ra từ độ dài hiện tại: lẻ → `+1`, chẵn → `−1`?
- [ ] Khi gãy mà `d == +1`, đặt `cur = 2` (đoạn mới), không phải `1`?
- [ ] Trả `-1` nếu không có đoạn độ dài `>= 2`?
- [ ] Đoạn là **liên tiếp** (mảng con)?
- [ ] Đã thử hai phần tử, toàn bằng nhau, `[4,5,6]`, `[1,2,3,2]`, `[3,2,3]`?
- [ ] Signature đúng `alternatingSubarray(self, nums: List[int]) -> int` và có `from typing import List`?